# ModuleList et register_buffer : enregistrer les états utiles

Étendre un module déjà compris sans confondre poids et référence fixe.

Nous lisons et exécutons les étapes dans l’ordre. Les cellules « Observer » montrent les valeurs et les formes ; elles ne prouvent pas la réussite des exercices.

Environnement vérifié : Python 3.11.9, PyTorch 2.10.0 CPU et TorchAudio 2.10.0 lorsque le chapitre le demande. Aucun modèle ni jeu de données n’est téléchargé pendant l’exécution.

[Revenir au chapitre](https://radiocodeur.github.io/quiz/cours.html?theme=pytorch#modules-et-buffers).


## Préparer le noyau

Choisissez un environnement CPU neuf. En cas de bibliothèque manquante, installez-la dans ce noyau, puis redémarrez-le. Les sorties affichent les versions réellement utilisées.

In [1]:
import os
import sys
import tempfile

# Choisir un noyau neuf pour garantir la vérification sur CPU.
os.environ["CUDA_VISIBLE_DEVICES"] = ""
try:
    import torch
except ImportError as error:
    raise RuntimeError("PyTorch manque. Installer torch==2.10.0 dans ce noyau, puis redémarrer et relancer.") from error
torch.set_num_threads(1)
torch.manual_seed(17)
print("Python", sys.version.split()[0], "— PyTorch", torch.__version__, "— CPU")
assert not torch.cuda.is_available(), "Redémarrer dans un noyau CPU neuf."
_quiz_initial_directory = os.getcwd()
_quiz_temporary_directory = tempfile.TemporaryDirectory(prefix="quiz-progressif-")
os.chdir(_quiz_temporary_directory.name)


Python 3.11.9 — PyTorch 2.10.0+cpu — CPU


In [2]:
# Cet outil sert uniquement à afficher les états des démonstrations.
# Il ne modifie pas les tenseurs et ne fait pas partie du modèle étudié.
def observer(nom, valeur):
    if isinstance(valeur, torch.Tensor):
        contenu = valeur.detach().cpu()
        print(nom, "forme", tuple(contenu.shape), "type", contenu.dtype,
              "requires_grad", valeur.requires_grad)
        if contenu.numel() <= 16:
            print("  valeurs :", contenu.tolist())
    elif valeur is None or isinstance(valeur, (int, float, bool, str, tuple)):
        print(nom, "=", valeur)
    elif isinstance(valeur, list) and len(valeur) <= 8:
        if all(isinstance(item,(int,float,bool,str)) for item in valeur):
            print(nom, "=", valeur)
        else:
            print(nom, ":",len(valeur),"éléments ; formes des tenseurs :",
                  [tuple(item.shape) for item in valeur if isinstance(item,torch.Tensor)])
    elif isinstance(valeur,dict):
        print(nom, ": clés",list(valeur)[:12])

def observer_entrainement(espace, montrer_poids=False):
    # Une copie évite de parcourir un dictionnaire global en mutation.
    vus = set()
    for nom, objet in list(espace.items()):
        if nom.startswith('_'):
            continue
        if isinstance(objet, torch.Tensor) and objet.is_leaf and objet.requires_grad:
            parametres = [(nom, objet)]
        elif isinstance(objet, torch.nn.Module):
            parametres = [(nom + '.' + cle, parametre)
                          for cle, parametre in list(objet.named_parameters())[:4]]
        else:
            continue
        for cle, parametre in parametres:
            if id(parametre) in vus:
                continue
            vus.add(id(parametre))
            if montrer_poids:
                observer(cle, parametre)
            observer(cle + '.grad', parametre.grad)


Cette extension vient après une première boucle et une sauvegarde simple. Nous voulons conserver une moyenne fixe et une collection de couches dans le modèle.

La moyenne [0,1 ; 1,5] est choisie pour la démonstration. Elle ne résulte pas d’un apprentissage. La couche reçoit ici les poids de l’identité pour rendre la sortie facile à vérifier.

| État | parameters() | state_dict() | Déplacement avec le module |
| --- | --- | --- | --- |
| Poids de la couche | Oui | Oui | Oui |
| Moyenne enregistrée en buffer | Non | Oui, par défaut | Oui |

Le buffer suit le module sans devenir une valeur ajustée par l’optimiseur.

## Démonstration 1 · Inspecter ce qui est enregistré

Relier chaque déclaration à sa présence dans le modèle.

### Préparer les bibliothèques

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [3]:
import torch
from torch import nn



### Définir la règle avant de l’utiliser

ModuleList conserve les couches comme sous-modules. Une simple liste Python ne les enregistre pas automatiquement.

register_buffer donne au module un tenseur nommé moyenne. Ce n’est pas un Parameter.

Nous centrons les entrées, puis appliquons les couches de la collection. La boucle ne crée pas de couche supplémentaire.

In [4]:
class ProjectionCentree(nn.Module):
    def __init__(self):
        super().__init__()
        self.couches = nn.ModuleList([nn.Linear(2, 2, bias=False)])
        self.register_buffer("moyenne", torch.tensor([0.1, 1.5]))

    def forward(self, x):
        centre = x - self.moyenne
        for couche in self.couches:
            centre = couche(centre)
        return centre



### Calculer l’état suivant

named_parameters, named_buffers et state_dict répondent à trois questions différentes.

In [5]:
modele = ProjectionCentree()


### Appliquer le contexte explicite

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [6]:
with torch.no_grad():
    modele.couches[0].weight.copy_(torch.eye(2))


### Calculer l’état suivant

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [7]:
x = torch.tensor([[0.2, 4.5]])


In [8]:
observer('x', x)

x forme (1, 2) type torch.float32 requires_grad False
  valeurs : [[0.20000000298023224, 4.5]]


### Lire les sorties

Ces impressions lisent les objets calculés. Elles n’entraînent pas le modèle et ne modifient pas les tenseurs.

In [9]:
print("sortie", modele(x).tolist())
print("parametres", [nom for nom, _ in modele.named_parameters()])
print("buffers", [nom for nom, _ in modele.named_buffers()])
print("sauvegarde", list(modele.state_dict()))

sortie [[0.10000000149011612, 3.0]]
parametres ['couches.0.weight']
buffers ['moyenne']
sauvegarde ['moyenne', 'couches.0.weight']


Le modèle sait retrouver sa couche et sa moyenne. Il confie uniquement les poids enregistrés à l’optimiseur.

## À vous de modifier le calcul

Les deux cellules suivantes sont prêtes à éditer. Si vous les exécutez sans les compléter, elles le signalent et ne prétendent pas que vous avez réussi. Une réponse fausse déclenche le contrôle correspondant.

### Exercice 1

Un buffer moyenne est enregistré. Doit-il apparaître dans state_dict ?

In [10]:
exo_modele = torch.nn.Linear(2,2)
exo_modele.register_buffer("moyenne",torch.zeros(2))
exo_sauvegarde = ...  # Remplacer les points par votre réponse.
if exo_sauvegarde is Ellipsis or exo_sauvegarde is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert exo_sauvegarde is ("moyenne" in exo_modele.state_dict())
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 1</summary>

Le buffer fait partie de l’état conservé, même s’il n’est pas appris comme un poids.

</details>

### Exercice 2

Ce buffer doit-il apparaître dans named_parameters ?

In [11]:
exo_modele = torch.nn.Linear(2,2)
exo_modele.register_buffer("moyenne",torch.zeros(2))
exo_parametre = ...  # Remplacer les points par votre réponse.
if exo_parametre is Ellipsis or exo_parametre is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert exo_parametre is ("moyenne" in dict(exo_modele.named_parameters()))
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 2</summary>

Un buffer enregistré est distinct d’un paramètre enregistré.

</details>

## Comparer avec les solutions

Ouvrez-les après avoir formulé votre réponse et votre explication. Les solutions sont également exécutées par le vérificateur du dépôt.

<details>
<summary>Solution de l’exercice 1</summary>

```python
exo_modele = torch.nn.Linear(2,2)
exo_modele.register_buffer("moyenne",torch.zeros(2))
exo_sauvegarde = True
if exo_sauvegarde is Ellipsis or exo_sauvegarde is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert exo_sauvegarde is ("moyenne" in exo_modele.state_dict())
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

<details>
<summary>Solution de l’exercice 2</summary>

```python
exo_modele = torch.nn.Linear(2,2)
exo_modele.register_buffer("moyenne",torch.zeros(2))
exo_parametre = False
if exo_parametre is Ellipsis or exo_parametre is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert exo_parametre is ("moyenne" in dict(exo_modele.named_parameters()))
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

In [12]:
os.chdir(_quiz_initial_directory)
_quiz_temporary_directory.cleanup()